## Session 04 Assignment: Semantic Search Engine
### Student Name: Chanchal Gore  
### Topic: Botany & Indian Medicinal Plants  
### Framework: Google GenAI SDK (`text-embedding-004`) & ChromaDB  


In [19]:
#Intialise the environment and load dependencies
import os
import numpy as np
import chromadb
from google import genai
from google.genai import types # Import native SDK types
from dotenv import load_dotenv

# Tell dotenv to look in the parent directory for the .env file
parent_directory_env = os.path.join("..", ".env")
load_dotenv(dotenv_path=parent_directory_env)

# Initialize the Gemini Client with explicit Developer API environment routing
client = genai.Client() 
print("[✓] Environment dependencies successfully initialized.")

[✓] Environment dependencies successfully initialized.


In [22]:
# 1. Define a knowledge base of 10 sentences regarding flora and traditional herbs
knowledge_base = [
    "Tulsi, or Holy Basil, is widely used in Indian households to cure coughs and common colds.",
    "Neem leaves are traditional Indian remedies known for killing harmful bacteria and clearing skin infections.",
    "Turmeric is a bright yellow spice used in India to heal wounds and reduce painful body swelling.",
    "Ashwagandha is an ancient Indian root that helps the brain reduce stress and calm anxiety.",
    "Amla, or Indian Gooseberry, is packed with Vitamin C and acts as an excellent booster for body immunity.",
    "Giloy is a climbing herb frequently consumed in India to purify blood and clear chronic fevers.",
    "Brahmi is a traditional creeping plant used in Ayurveda to sharpen memory and improve mental focus.",
    "Aloe Vera stores cooling gel inside its thick leaves and is applied to soothe skin burns.",
    "Ginger root is an essential kitchen herb used across India to relieve upset stomachs and nausea.",
    "Arjuna bark extract is a famous traditional remedy used to strengthen heart muscles and control blood pressure."
]

doc_ids = [f"indian_plant_{i}" for i in range(len(knowledge_base))]

print("[*] Encoding 10 Indian plant facts into vectors...")

# 2. EMBED: Using 'gemini-embedding-001' which matches the v1beta endpoint routing perfectly
embeddings = []
for text in knowledge_base:
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )
    embeddings.append(response.embeddings[0].values)

# 3. STORE: Save the vectors in a local vector database
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# 4. Using a new collection name to avoid conflict with the previous run
collection = chroma_client.get_or_create_collection(name="expanded_indian_herbs_db")

collection.add(
    ids=doc_ids,
    embeddings=embeddings,
    documents=knowledge_base
)
print(f"[✓] Successfully stored {collection.count()} Indian plant records!")

[*] Encoding 10 Indian plant facts into vectors...
[✓] Successfully stored 10 Indian plant records!


In [27]:
# Semantic queries to test the knowledge base
test_queries = [
    "What home remedy can help me stop coughing?",
    "Is there an herb that can protect my body from falling sick?",
    "Which traditional root should I take to relax my mind from stress?",
    "What natural ingredient can help clear up pimples or skin rashes?",
    "Are there any native remedies known for boosting memory and brain power?"
]

print("          GEMINI SEMANTIC RETRIEVAL MATRIX               ")

for query_text in test_queries:
    # 1. Turn the search question into a numeric vector
    query_vector = client.models.embed_content(
        model="gemini-embedding-001",
        contents=query_text
    ).embeddings[0].values
    
    # 2. fetch the top 3 similar sentences
    results = collection.query(
        query_embeddings=[query_vector], 
        n_results=3  # <-- This tells ChromaDB to pull exactly 3 matches
    )
    
    print(f"Your Question: '{query_text}'")
    print("-" * 55)
    
    # 3. Loop through and print all 3 returned matching sentences
    for index, doc in enumerate(results['documents'][0]):
        print(f"  [Match {index + 1}] {doc}")
    print("\n" + "="*55 + "\n")

          GEMINI SEMANTIC RETRIEVAL MATRIX               
Your Question: 'What home remedy can help me stop coughing?'
-------------------------------------------------------
  [Match 1] Tulsi, or Holy Basil, is widely used in Indian households to cure coughs and common colds.
  [Match 2] Amla, or Indian Gooseberry, is packed with Vitamin C and acts as an excellent booster for body immunity.
  [Match 3] Ginger root is an essential kitchen herb used across India to relieve upset stomachs and nausea.


Your Question: 'Is there an herb that can protect my body from falling sick?'
-------------------------------------------------------
  [Match 1] Tulsi, or Holy Basil, is widely used in Indian households to cure coughs and common colds.
  [Match 2] Amla, or Indian Gooseberry, is packed with Vitamin C and acts as an excellent booster for body immunity.
  [Match 3] Giloy is a climbing herb frequently consumed in India to purify blood and clear chronic fevers.


Your Question: 'Which traditio

In [26]:
# Cosine similarity function to compare two vectors
def compute_cosine_similarity(vec_a, vec_b):
    array_a, array_b = np.array(vec_a), np.array(vec_b)
    return np.dot(array_a, array_b) / (np.linalg.norm(array_a) * np.linalg.norm(array_b))

# Testing closely related vs completely different plant sentences
pairs_to_test = [
    (knowledge_base[0], knowledge_base[1]),  # Neem vs Holy Basil (Both Herbs)
    (knowledge_base[4], knowledge_base[7])   # Photosynthesis vs Hydroponics (Both Science)
]

for text_1, text_2 in pairs_to_test:
    # 1. Generate structural embeddings for both sentences
    res = client.models.embed_content(model="gemini-embedding-001", contents=[text_1, text_2])
    v1 = res.embeddings[0].values
    v2 = res.embeddings[1].values
    
    # 2. Run the math comparison
    score = compute_cosine_similarity(v1, v2)
    
    print(f"A: {text_1[:45]}...")
    print(f"B: {text_2[:45]}...")
    print(f"Cosine Similarity Score: {score:.4f}\n")

A: Tulsi, or Holy Basil, is widely used in India...
B: Neem leaves are traditional Indian remedies k...
Cosine Similarity Score: 0.6846

A: Amla, or Indian Gooseberry, is packed with Vi...
B: Aloe Vera stores cooling gel inside its thick...
Cosine Similarity Score: 0.6012



## Part 3 — Reflection Questions

### 1. Did any query return a wrong or surprising result? What caused it?
Overall, the semantic search performed remarkably well. For example, searching for "clear up pimples or skin rashes" correctly pulled up Neem and Aloe Vera as top matches without exact keyword matching. However, a slightly surprising result was seeing Amla or Tulsi show up as the third match for a skin query. This happens because all these plants exist within a shared "Ayurvedic medicinal herbs" contextual cluster in the high-dimensional vector space, causing general botanical medicine concepts to slightly overlap spatially.

### 2. What would happen if you used a model with only 384 dimensions instead of Gemini's 768?
If we switched to a smaller 384-dimension embedding model, the vector representation space would be compressed. While this would make distance calculations inside ChromaDB faster and save local storage space, the search engine would lose its fine-grained nuance tracking. It would likely struggle to separate specific clinical use cases—potentially mixing up distinct remedies like heart health (Arjuna Bark) and memory enhancement (Brahmi) because it lacks enough unique geometric axes to keep them distinct.

### 3. Why is it critical to use the same embedding model for both documents and queries?
Every individual embedding model sets up its own unique mathematical universe with distinct coordinate rules. Gemini's vector dimensions map semantic concepts to entirely different layout numbers than an OpenAI, Cohere, or HuggingFace model. If you process your Indian plant facts using Gemini but attempt to search through them using a query vector generated by a different model, the spatial alignment fails completely. The search coordinates would point to random positions, silently breaking your retrieval system without throwing a visible code error.